# EXP_10: Passage-Level LIME Explainability

Identifies which retrieved evidence passages most influenced each generated medical answer
using **Local Interpretable Model-agnostic Explanations (LIME)**.

**Method:** Each retrieved passage is treated as a binary interpretable feature (present / absent).
For every question, we create perturbations by masking individual passages, regenerate the answer
via Groq LLaMA 3.3 70B, and measure how answer quality changes. A local linear surrogate model
then ranks passages by their influence on answer quality.

**Target architectures:**
- Evidence-Graded RAG (5 graded passages per question — the top-5 actually fed to the generator, not the wider k=8 candidate pool)
- Query Decomposition RAG (5 passages per question)

**Important:** For Evidence-Graded RAG, we use `graded_contexts` (the 5 passages the generator
actually saw), not `retrieved_contexts` (the 8-passage candidate pool). Attributing influence
to passages the LLM never received would be methodologically invalid.

In [ ]:
import sys
sys.path.append("..")

import os
import time
import json
import numpy as np
import pandas as pd
from ast import literal_eval
from datetime import datetime
from itertools import combinations

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from rouge_score import rouge_scorer
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

import config

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", font_scale=1.1)

RESULTS_DIR = config.RESULTS_EVALSETS_DIR
DEEPEVAL_DIR = config.RESULTS_DEEPEVAL_DIR
FIGURES_DIR = config.RESULTS_FIGURES_DIR

## Configuration

In [ ]:
ARCHITECTURES = {
    "Evidence-Graded RAG": {
        "eval_file": "evidence_graded_rag_minilm_chroma_20260516_164252.csv",
        "faithfulness_file": "evidence_graded_rag_minilm_faithfulness_20260516_164252.csv",
        "correctness_file": "evidence_graded_rag_minilm_answer_correctness_20260516_164252.csv",
    },
    "Query Decomposition RAG": {
        "eval_file": "decomposition_rag_minilm_k_5_20260518_181816.csv",
        "faithfulness_file": "decomposition_rag_minilm_faithfulness_20260518_181816.csv",
        "correctness_file": "decomposition_rag_minilm_answer_correctness_20260518_181816.csv",
    },
}

# LIME parameters
NUM_PERTURBATIONS = 50       # perturbations per question (beyond leave-one-out)
RIDGE_ALPHA = 1.0            # regularisation for surrogate model
GROQ_DELAY = config.PARALLEL_DELAY_SECONDS  # rate limit between LLM calls

# Prompts matching the original experiments
EVIDENCE_GRADED_PROMPT = """You are a biomedical research assistant. Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Instructions:
- Synthesize an answer from the provided context using evidence-based reasoning.
- Draw logical inferences and conclusions from the evidence when a direct statement is not available.
- Cite specific findings, statistics, or conclusions from the context to support your answer.
- Do NOT refuse to answer or state that the context is insufficient. Use whatever relevant evidence is available.

Answer:"""

DECOMPOSITION_PROMPT = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

PROMPTS = {
    "Evidence-Graded RAG": EVIDENCE_GRADED_PROMPT,
    "Query Decomposition RAG": DECOMPOSITION_PROMPT,
}

## Groq Client & Scoring Utilities

In [ ]:
class GroqKeyRotator:
    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"GroqKeyRotator: {len(self.api_keys)} API key(s)")

    def get_llm(self):
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
            temperature=0,
            max_tokens=900,
        )

    def rotate(self):
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)


rotator = GroqKeyRotator()
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def score_answer(generated: str, golden: str) -> float:
    """ROUGE-L F1 between generated and golden answer."""
    return scorer.score(golden, generated)["rougeL"].fmeasure


def generate_answer(question: str, passages: list[str], prompt_template: str) -> str:
    """Call Groq to generate an answer from the given passages."""
    context = "\n\n---\n\n".join(passages) if passages else "No context available."
    prompt = PromptTemplate(
        template=prompt_template, input_variables=["context", "question"]
    )
    llm = rotator.get_llm()
    chain = prompt | llm
    try:
        result = chain.invoke({"context": context, "question": question})
        return result.content
    except Exception as e:
        print(f"  LLM error: {e}, rotating key...")
        rotator.rotate()
        time.sleep(GROQ_DELAY * 2)
        llm = rotator.get_llm()
        chain = prompt | llm
        result = chain.invoke({"context": context, "question": question})
        return result.content

## Load Eval Datasets

In [ ]:
datasets = {}
for arch_name, files in ARCHITECTURES.items():
    df = pd.read_csv(RESULTS_DIR / files["eval_file"])

    # Use graded_contexts (the 5 passages actually fed to the generator) for
    # Evidence-Graded RAG; use retrieved_contexts for architectures without grading.
    if "graded_contexts" in df.columns:
        df["graded_contexts"] = df["graded_contexts"].apply(literal_eval)
        df["passages_for_xai"] = df["graded_contexts"]
    else:
        df["retrieved_contexts"] = df["retrieved_contexts"].apply(literal_eval)
        df["passages_for_xai"] = df["retrieved_contexts"]

    df["num_passages"] = df["passages_for_xai"].apply(len)

    # Load faithfulness scores
    faith_df = pd.read_csv(DEEPEVAL_DIR / files["faithfulness_file"])
    faith_col = [c for c in faith_df.columns if "Faithfulness" in c or "faithfulness" in c.lower()]
    if faith_col:
        if "question_idx" in faith_df.columns:
            df = df.merge(faith_df[["question_idx", faith_col[0]]], on="question_idx", how="left")
        else:
            df["faithfulness"] = faith_df[faith_col[0]].values[:len(df)]
    
    # Load correctness scores
    corr_df = pd.read_csv(DEEPEVAL_DIR / files["correctness_file"])
    corr_col = [c for c in corr_df.columns if "Correctness" in c or "correctness" in c.lower()]
    if corr_col:
        if "question_idx" in corr_df.columns:
            df = df.merge(corr_df[["question_idx", corr_col[0]]], on="question_idx", how="left", suffixes=("", "_corr"))
        else:
            df["answer_correctness"] = corr_df[corr_col[0]].values[:len(df)]

    datasets[arch_name] = df
    ctx_source = "graded_contexts" if "graded_contexts" in df.columns else "retrieved_contexts"
    print(f"{arch_name}: {len(df)} questions, {df['num_passages'].iloc[0]} passages each (from {ctx_source})")
    print(f"  Columns: {list(df.columns)}")
    print()

## LIME Perturbation Engine

For each question:
1. **Leave-one-out** perturbations: mask each passage individually (k perturbations)
2. **Random subset** perturbations: randomly include/exclude passages to build a richer local dataset
3. **Full context** baseline: score with all passages present
4. **No context** baseline: score with no passages
5. Fit a Ridge regression on the binary perturbation matrix → scores
6. Passage influence = regression coefficients (higher = more influential)

In [ ]:
def generate_perturbation_matrix(n_passages: int, n_random: int = 30, seed: int = 42) -> np.ndarray:
    """Create binary perturbation matrix.
    
    Rows are perturbations, columns are passages.
    Always includes: all-ones (full context), all-zeros (no context),
    leave-one-out (k rows), and n_random random subsets.
    """
    rng = np.random.RandomState(seed)
    rows = []

    # Full context
    rows.append(np.ones(n_passages, dtype=int))

    # No context
    rows.append(np.zeros(n_passages, dtype=int))

    # Leave-one-out
    for i in range(n_passages):
        row = np.ones(n_passages, dtype=int)
        row[i] = 0
        rows.append(row)

    # Random subsets
    for _ in range(n_random):
        row = rng.randint(0, 2, size=n_passages)
        if row.sum() == 0 or row.sum() == n_passages:
            row[rng.randint(0, n_passages)] ^= 1
        rows.append(row)

    return np.array(rows)


def compute_lime_scores(
    question: str,
    passages: list[str],
    golden_answer: str,
    prompt_template: str,
    n_random: int = 30,
    seed: int = 42,
) -> dict:
    """Run LIME for a single question. Returns passage influence scores."""
    n_passages = len(passages)
    perturbation_matrix = generate_perturbation_matrix(n_passages, n_random, seed)
    scores = []

    for i, mask in enumerate(perturbation_matrix):
        active_passages = [p for p, m in zip(passages, mask) if m == 1]
        answer = generate_answer(question, active_passages, prompt_template)
        s = score_answer(answer, golden_answer)
        scores.append(s)
        time.sleep(GROQ_DELAY)

    scores = np.array(scores)

    # Fit local surrogate
    # Weight samples by proximity to full-context instance (cosine distance in binary space)
    full_context = perturbation_matrix[0]
    distances = np.sqrt(((perturbation_matrix - full_context) ** 2).sum(axis=1))
    kernel_width = np.sqrt(n_passages) * 0.75
    weights = np.exp(-(distances ** 2) / (kernel_width ** 2))

    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(perturbation_matrix, scores, sample_weight=weights)

    return {
        "influence_scores": model.coef_.tolist(),
        "intercept": float(model.intercept_),
        "full_context_score": float(scores[0]),
        "no_context_score": float(scores[1]),
        "r_squared": float(model.score(perturbation_matrix, scores, sample_weight=weights)),
        "n_perturbations": len(scores),
    }

## Run LIME Across Both Architectures

This cell processes all 200 questions per architecture. Each question requires
~37–42 LLM calls (2 baselines + k leave-one-out + 30 random). With a 4-second delay
between calls, expect ~3–4 hours per architecture.

Results are checkpointed after each question so progress is not lost.

In [ ]:
for arch_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Processing: {arch_name}")
    print(f"{'='*60}")

    prompt_template = PROMPTS[arch_name]
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"

    # Resume from checkpoint if exists
    if output_file.exists():
        existing = pd.read_csv(output_file)
        done_indices = set(existing["question_idx"].values)
        results = existing.to_dict("records")
        print(f"  Resuming from checkpoint: {len(done_indices)} questions already done")
    else:
        done_indices = set()
        results = []

    for row_idx, row in df.iterrows():
        q_idx = row["question_idx"]
        if q_idx in done_indices:
            continue

        question = row["question"]
        passages = row["passages_for_xai"]
        golden_answer = row["golden_answer"]

        print(f"  [{len(results)+1}/{len(df)}] Q{q_idx}: {question[:80]}...")

        try:
            lime_result = compute_lime_scores(
                question, passages, golden_answer, prompt_template,
                n_random=NUM_PERTURBATIONS - len(passages) - 2,  # total = n_random + k + 2
            )

            record = {
                "question_idx": q_idx,
                "question": question,
                "num_passages": len(passages),
                "full_context_score": lime_result["full_context_score"],
                "no_context_score": lime_result["no_context_score"],
                "r_squared": lime_result["r_squared"],
                "intercept": lime_result["intercept"],
                "n_perturbations": lime_result["n_perturbations"],
            }

            # Store per-passage influence scores
            for p_idx, score in enumerate(lime_result["influence_scores"]):
                record[f"passage_{p_idx}_influence"] = score

            # Store passage ranking (most influential first)
            ranking = np.argsort(lime_result["influence_scores"])[::-1]
            record["passage_ranking"] = json.dumps(ranking.tolist())
            record["top1_passage_idx"] = int(ranking[0])
            record["top3_passage_indices"] = json.dumps(ranking[:3].tolist())

            results.append(record)

            # Checkpoint every 5 questions
            if len(results) % 5 == 0:
                pd.DataFrame(results).to_csv(output_file, index=False)
                print(f"    Checkpointed at {len(results)} questions")

        except Exception as e:
            print(f"    ERROR on Q{q_idx}: {e}")
            rotator.rotate()
            time.sleep(GROQ_DELAY * 3)
            continue

    # Final save
    result_df = pd.DataFrame(results)
    result_df.to_csv(output_file, index=False)
    print(f"\n  Saved {len(result_df)} results to {output_file}")

## Analysis: Passage Influence Distribution

In [ ]:
for arch_name in ARCHITECTURES:
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        print(f"Skipping {arch_name}: no results yet")
        continue

    lime_df = pd.read_csv(output_file)
    influence_cols = [c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")]
    n_passages = len(influence_cols)

    print(f"\n{'='*50}")
    print(f"{arch_name} — {len(lime_df)} questions, {n_passages} passages")
    print(f"{'='*50}")

    # Summary statistics
    influence_matrix = lime_df[influence_cols].values
    mean_influence = influence_matrix.mean(axis=0)
    print(f"\nMean influence per passage position:")
    for i, m in enumerate(mean_influence):
        bar = "█" * int(abs(m) * 50)
        sign = "+" if m >= 0 else "-"
        print(f"  Passage {i}: {sign}{abs(m):.4f} {bar}")

    print(f"\nSurrogate model R²: {lime_df['r_squared'].mean():.3f} (mean), {lime_df['r_squared'].median():.3f} (median)")
    print(f"Full-context ROUGE-L: {lime_df['full_context_score'].mean():.3f}")
    print(f"No-context ROUGE-L: {lime_df['no_context_score'].mean():.3f}")

## Visualisation: LIME Influence Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax_idx, arch_name in enumerate(ARCHITECTURES):
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        continue

    lime_df = pd.read_csv(output_file)
    influence_cols = [c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")]
    influence_matrix = lime_df[influence_cols].values

    # Sort questions by total absolute influence for visual clarity
    sort_idx = np.argsort(np.abs(influence_matrix).sum(axis=1))[::-1]
    display_matrix = influence_matrix[sort_idx[:50]]  # top 50 questions

    ax = axes[ax_idx]
    im = ax.imshow(display_matrix, aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3)
    ax.set_xlabel("Passage Index")
    ax.set_ylabel("Question (sorted by total influence)")
    ax.set_title(f"{arch_name}\nPassage Influence Heatmap")
    ax.set_xticks(range(len(influence_cols)))
    ax.set_xticklabels([f"P{i}" for i in range(len(influence_cols))])
    plt.colorbar(im, ax=ax, label="LIME Influence Score", shrink=0.8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "lime_influence_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lime_influence_heatmap.png")

## Visualisation: Mean Influence by Passage Position

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, arch_name in enumerate(ARCHITECTURES):
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        continue

    lime_df = pd.read_csv(output_file)
    influence_cols = [c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")]
    influence_matrix = lime_df[influence_cols].values

    mean_influence = influence_matrix.mean(axis=0)
    std_influence = influence_matrix.std(axis=0)

    ax = axes[ax_idx]
    colours = ["#2ecc71" if v >= 0 else "#e74c3c" for v in mean_influence]
    bars = ax.bar(range(len(mean_influence)), mean_influence, yerr=std_influence,
                  color=colours, capsize=3, edgecolor="white", linewidth=0.5)
    ax.axhline(y=0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Passage Position")
    ax.set_ylabel("Mean LIME Influence")
    ax.set_title(f"{arch_name}")
    ax.set_xticks(range(len(mean_influence)))
    ax.set_xticklabels([f"P{i}" for i in range(len(mean_influence))])

plt.suptitle("Mean Passage Influence by Position (LIME)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lime_mean_influence_by_position.png", dpi=150, bbox_inches="tight")
plt.show()

## Visualisation: Top-1 Passage Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, arch_name in enumerate(ARCHITECTURES):
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        continue

    lime_df = pd.read_csv(output_file)
    n_passages = len([c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")])

    ax = axes[ax_idx]
    top1_counts = lime_df["top1_passage_idx"].value_counts().sort_index()
    ax.bar(top1_counts.index, top1_counts.values, color="steelblue", edgecolor="white")
    ax.set_xlabel("Passage Index")
    ax.set_ylabel("Times Ranked #1")
    ax.set_title(f"{arch_name}")
    ax.set_xticks(range(n_passages))
    ax.set_xticklabels([f"P{i}" for i in range(n_passages)])

plt.suptitle("Which Passage Is Most Influential? (Top-1 Distribution)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lime_top1_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Correlation: LIME Influence vs Faithfulness

In [ ]:
for arch_name, df in datasets.items():
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        continue

    lime_df = pd.read_csv(output_file)
    influence_cols = [c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")]

    # Max influence per question = how much the best passage contributes
    lime_df["max_influence"] = lime_df[influence_cols].max(axis=1)
    lime_df["influence_spread"] = lime_df[influence_cols].max(axis=1) - lime_df[influence_cols].min(axis=1)

    # Merge with faithfulness
    faith_cols = [c for c in df.columns if "faithfulness" in c.lower() or "Faithfulness" in c]
    if faith_cols:
        merged = lime_df.merge(
            df[["question_idx"] + faith_cols],
            on="question_idx", how="inner"
        )
        faith_col = faith_cols[0]

        print(f"\n{arch_name}:")
        corr_max = merged["max_influence"].corr(merged[faith_col])
        corr_spread = merged["influence_spread"].corr(merged[faith_col])
        print(f"  Pearson(max_influence, faithfulness) = {corr_max:.3f}")
        print(f"  Pearson(influence_spread, faithfulness) = {corr_spread:.3f}")

## Summary Table

In [ ]:
summary_rows = []
for arch_name in ARCHITECTURES:
    output_file = RESULTS_DIR / f"lime_passage_scores_{arch_name.lower().replace(' ', '_')}.csv"
    if not output_file.exists():
        continue
    lime_df = pd.read_csv(output_file)
    influence_cols = [c for c in lime_df.columns if c.startswith("passage_") and c.endswith("_influence")]
    influence_matrix = lime_df[influence_cols].values

    summary_rows.append({
        "Architecture": arch_name,
        "N Questions": len(lime_df),
        "N Passages": len(influence_cols),
        "Mean R²": f"{lime_df['r_squared'].mean():.3f}",
        "Full-Ctx ROUGE-L": f"{lime_df['full_context_score'].mean():.3f}",
        "No-Ctx ROUGE-L": f"{lime_df['no_context_score'].mean():.3f}",
        "Mean Max Influence": f"{influence_matrix.max(axis=1).mean():.4f}",
        "Mean Influence Spread": f"{(influence_matrix.max(axis=1) - influence_matrix.min(axis=1)).mean():.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df